# **Notebook 06: Model Monitoring + Drift Detection**
## **Production Fraud Detection Platform on AWS**

---

### **Project Context**

This is the final notebook in our 6-notebook series.
We now have a deployed V3 XGBoost model (AUC 0.9622)
that we are proud of. But in production, the world
changes — fraudsters adapt, new card types emerge,
spending patterns shift. A model that performs
brilliantly on day 1 can silently degrade by day 90.

This notebook answers the question every production
ML engineer must answer:

**"How do we know when our model stops working?"**

---

### **What is Model Monitoring?**

| Concept | Definition |
|---|---|
| **Data drift** | Input feature distributions change over time |
| **Concept drift** | The relationship between features and fraud changes |
| **Score drift** | Model output scores shift without retraining |
| **Model degradation** | Precision and recall drop below acceptable thresholds |

---

### **Our Approach**

We simulate 6 months of production traffic using the
IEEE-CIS dataset, then inject realistic drift scenarios:

| Month | Scenario |
|---|---|
| Month 1–2 | Normal traffic — baseline period |
| Month 3 | New fraud pattern — small amounts |
| Month 4 | Compromised card network — card1 drift |
| Month 5 | Email domain shift — new fraud domains |
| Month 6 | Full concept drift — model degrades |

For each month we measure feature drift (KS test + PSI),
score drift, and model performance to build a complete
monitoring dashboard.

---

### **Notebook Structure**

| Cell | Topic |
|---|---|
| Cell 1 | Setup + load V3 model and data |
| Cell 2 | Baseline statistics (Month 1–2) |
| Cell 3 | Simulate drift scenarios (Month 3–6) |
| Cell 4 | Feature drift detection (KS test + PSI) |
| Cell 5 | Score drift + model degradation |
| Cell 6 | Alerting system |
| Cell 7 | Full project conclusion |

## **Setup + Load V3 Model and Data**

### What we do
Load the V3 XGBoost model and df_features_v3.csv
from S3. We split the data into monthly batches
to simulate 6 months of production traffic.

### Why it matters
Real monitoring requires a time axis — we need
batches of transactions to compare against the
baseline, not a single snapshot.

In [ ]:
# ============================================================
# CELL 1: Setup + Load V3 Model and Data
# ============================================================

import pandas as pd
import numpy as np
import boto3
import xgboost as xgb
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

print("=" * 55)
print("  NOTEBOOK 06 — MODEL MONITORING")
print("  Drift Detection + Alerting")
print("=" * 55)

BUCKET = "fraud-detection-mlproject-armand"
REGION = "us-east-1"
s3     = boto3.client('s3', region_name=REGION)

# ── STEP 1: Load V3 model ─────────────────────────────────────
print("\nStep 1: Loading V3 XGBoost model...")

os.makedirs('/tmp/v3', exist_ok=True)

for key, name in [
    ('models/v3/xgb_model_v3fix.json',     'model.json'),
    ('models/v3/feature_names_v3fix.json', 'features.json'),
]:
    s3.download_file(BUCKET, key, f'/tmp/v3/{name}')

model = xgb.Booster()
model.load_model('/tmp/v3/model.json')

with open('/tmp/v3/features.json') as f:
    FEATURE_NAMES = json.load(f)

THRESHOLD = 0.87
print(f"   Model loaded!")
print(f"   Features  : {len(FEATURE_NAMES)}")
print(f"   Threshold : {THRESHOLD}")

# ── STEP 2: Load df_features_v3 ───────────────────────────────
print("\nStep 2: Loading df_features_v3 from S3...")

if not os.path.exists('/tmp/df_features_v3.csv'):
    print("   Downloading (1.3GB — ~2 min)...")
    s3.download_file(
        BUCKET,
        'processed-data/df_features_v3.csv',
        '/tmp/df_features_v3.csv'
    )
else:
    print("   File already local — skipping download!")

df = pd.read_csv('/tmp/df_features_v3.csv')
print(f"   Shape  : {df.shape}")
print(f"   Fraud  : {df['isFraud'].sum():,} "
      f"({df['isFraud'].mean()*100:.2f}%)")

# ── STEP 3: Create monthly batches ────────────────────────────
print("\nStep 3: Creating monthly batches...")

# Simulate 6 months by splitting data into 6 equal chunks
# Sort by TransactionDT to preserve time order
df = df.sort_values('TransactionDT').reset_index(drop=True)
n  = len(df)
month_size = n // 6

months = {}
for i in range(6):
    start = i * month_size
    end   = (i + 1) * month_size if i < 5 else n
    months[f'month_{i+1}'] = df.iloc[start:end].copy()

print(f"\n   {'Month':<10} {'Rows':>8} "
      f"{'Fraud':>8} {'Fraud%':>8}")
print(f"   {'-'*38}")
for name, batch in months.items():
    fraud_rate = batch['isFraud'].mean() * 100
    print(f"   {name:<10} {len(batch):>8,} "
          f"{batch['isFraud'].sum():>8,} "
          f"{fraud_rate:>7.2f}%")

# ── STEP 4: Score baseline ────────────────────────────────────
print("\nStep 4: Scoring baseline (Month 1)...")

def score_batch(batch, model, feature_names):
    X = batch[feature_names].fillna(0).astype(np.float32)
    dmat = xgb.DMatrix(
        pd.DataFrame(X, columns=feature_names)
    )
    return model.predict(dmat)

baseline    = months['month_1']
base_scores = score_batch(
    baseline, model, FEATURE_NAMES
)
base_preds  = (base_scores >= THRESHOLD).astype(int)

from sklearn.metrics import (
    roc_auc_score, f1_score,
    precision_score, recall_score
)

base_auc  = roc_auc_score(
    baseline['isFraud'], base_scores
)
base_f1   = f1_score(
    baseline['isFraud'], base_preds
)
base_prec = precision_score(
    baseline['isFraud'], base_preds
)
base_rec  = recall_score(
    baseline['isFraud'], base_preds
)

print(f"\n   Baseline (Month 1) performance:")
print(f"   AUC-ROC   : {base_auc:.4f}")
print(f"   F1-Score  : {base_f1:.4f}")
print(f"   Precision : {base_prec:.4f}")
print(f"   Recall    : {base_rec:.4f}")
print(f"   Avg score : {base_scores.mean():.4f}")

# Store globals
MONTHS       = months
BASE_SCORES  = base_scores
BASELINE     = baseline
BASE_METRICS = {
    'auc': base_auc, 'f1': base_f1,
    'precision': base_prec, 'recall': base_rec,
    'avg_score': base_scores.mean()
}

print(f"\n{'='*55}")
print(f"Setup complete!")
print(f"   Model     : V3 XGBoost (AUC 0.9622)")
print(f"   Data      : {n:,} transactions")
print(f"   Months    : 6 batches of "
      f"~{month_size:,} rows each")
print(f"{'='*55}")

  NOTEBOOK 06 — MODEL MONITORING
  Drift Detection + Alerting

Step 1: Loading V3 XGBoost model...
   Model loaded!
   Features  : 626
   Threshold : 0.87

Step 2: Loading df_features_v3 from S3...
   File already local — skipping download!
